# Create National Computational Infrastructure (NCI Australia) Awards

Creates awards for the **in-kind** merit allocations of computing time (kSU, kilo service units)
on NCI Australia's Gadi supercomputer, from the two schemes whose outcomes NCI publishes per project:
- **NCMAS** (National Computational Merit Allocation Scheme, run by NCI with Pawsey): outcomes
  2021-2026 as the per-year CSV export at `my.nci.org.au/mancini/ncmas/<year>/outcomes/csv`.
  **794 awards** = every NCMAS project with a Gadi allocation. 106 projects allocated only on
  another facility (Pawsey Magnus/Setonix, Monash MASSIVE, UQ FlashLite) are not NCI's allocation
  and are left out (Pawsey is not an OpenAlex funder); other-facility kSU of mixed projects are
  listed in the description.
- **ANUMAS** (ANU Merit Allocation Scheme, the merit allocation of ANU's partner share of NCI):
  outcomes 2020-2026 at `anumas.nci.org.au/outcomes/<year>/`. **173 awards** (one per NCI project
  code, 442 yearly allocations). Titles are published only for 2020-2021.

Total **967 awards** (local run 2026-10-01). Not ingested: NCMAS <= 2020 (Raijin era; only in the
Wayback Machine, 2020 not retrievable) and other partner shares (no public per-project lists).

**robots.txt:** nci.org.au disallows ClaudeBot / Claude-Web / anthropic-ai; my.nci.org.au and
anumas.nci.org.au serve no robots.txt (404 = no restrictions under RFC 9309). The www site is not
fetched.

**Prerequisites:** run `scripts/local/nci_australia_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/nci_australia/nci_australia_projects.parquet` (1.4 shrink guard).

**`funder_award_id` (2.1.1):** papers cite NCMAS proposals as "NCMAS-2024-59" and NCI projects by
code ("Project y89", "project fk5"). NCMAS: the Project ID (`NCMAS-2026-4`), one award per
proposal-year; 9 citation stubs collapse exactly. ANUMAS: the NCI project code (`y89`, lower-cased,
annotations such as "(12720)" / "(NEW)" stripped), one award per code; a first-year "NEW" project
(code not yet assigned) is merged into the code that first appears the next year under the same
lead CI when unique (26 of 80), else kept as `ANUMAS-<year>-NEW-<lead-slug>` (54).

**Amounts / dates:** in-kind, so `amount`/`currency` are NULL; the kSU allocations are spelled out
in `description`. `funding_type = 'grant'` (same as NWO computing-time awards; consistent across the
w20 compute funders). Allocations are per calendar year: `start_year` = first year, `end_year` =
last year; no dates are published.

**lead_investigator:** the Lead CI (split in Python, runbook 2.4.1), institution as affiliation
(country AU; ANUMAS: Australian National University).

**Funder details (Path A, F4320* Crossref-registered):** funder_id `4320312169` (National
Computational Infrastructure, AU; ror 04yx6dh41; doi 10.13039/100010582).

**Priority:** `554` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nci_australia_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/nci_australia/nci_australia_projects.parquet`;

In [ ]:
%sql
SELECT scheme, first_year, COUNT(*) as n FROM openalex.awards.nci_australia_raw GROUP BY 1, 2 ORDER BY 1, 2;

## Step 1.5: Inspect raw data

In [ ]:
%sql
DESCRIBE openalex.awards.nci_australia_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.nci_australia_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320312169 is a Crossref-registered funder, so it MUST resolve to exactly 1 row in
`openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320312169;

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320312169 must have exactly one row in openalex.funders.funders') AS funder_ok
FROM openalex.funders.funders
WHERE funder_id = 4320312169;

## Step 2: Create NCI Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nci_australia_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320312169  -- National Computational Infrastructure (NCI Australia)
),

g AS (
    SELECT r.*, TRIM(r.funder_award_id) AS award_id
    FROM openalex.awards.nci_australia_raw r
    WHERE r.funder_award_id IS NOT NULL AND TRIM(r.funder_award_id) != ''
),

lead AS (
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        award_id,
        CASE WHEN NULLIF(TRIM(lead_family_name), '') IS NOT NULL THEN
        named_struct(
            'given_name', NULLIF(TRIM(lead_given_name), ''),
            'family_name', NULLIF(TRIM(lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(institution), ''),
                'country', 'AU',
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead_investigator
    FROM g
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_id)))) % 9000000000 as id,
    COALESCE(
        NULLIF(TRIM(g.title), ''),
        CASE WHEN g.award_id LIKE 'ANUMAS-%'
             THEN CONCAT('NCI Gadi compute allocation (ANU Merit Allocation Scheme ', g.first_year, '): new project led by ', g.lead_ci)
             ELSE CONCAT('NCI Gadi compute allocation (ANU Merit Allocation Scheme): project ', g.award_id) END
    ) as display_name,
    CONCAT_WS(' ',
        'In-kind allocation of computing time on the NCI Australia Gadi supercomputer, in kSU (kilo service units; no monetary award).',
        CONCAT(g.allocation_text, '.'),
        CASE WHEN g.scheme = 'ANUMAS' AND NULLIF(TRIM(g.school), '') IS NOT NULL THEN CONCAT('Lead CI unit: ', TRIM(g.school), '.') END,
        CASE WHEN NULLIF(TRIM(g.fields_of_research), '') IS NOT NULL THEN CONCAT('Fields of research: ', TRIM(g.fields_of_research)) END
    ) as description,
    f.funder_id,
    g.award_id as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'grant' as funding_type,
    CASE g.scheme
        WHEN 'NCMAS' THEN 'National Computational Merit Allocation Scheme (NCMAS)'
        WHEN 'ANUMAS' THEN 'ANU Merit Allocation Scheme (ANUMAS, ANU partner share of NCI)'
    END as funder_scheme,
    'nci_merit_allocations' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.first_year AS INT) as start_year,
    TRY_CAST(g.last_year AS INT) as end_year,
    l.lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN l.lead_investigator IS NOT NULL THEN array(l.lead_investigator) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_id)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
JOIN lead l ON l.award_id = g.award_id
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (2.1.1): NCMAS proposal ids, NCI project codes, or ANUMAS NEW keys; ids unique; no rows lost.
SELECT
    assert_true(COUNT_IF(NOT funder_award_id RLIKE '^(NCMAS-20[0-9]{2}-[0-9]+|[a-z]{1,3}[0-9]{1,3}|ANUMAS-20[0-9]{2}-NEW-[a-z0-9-]+)$') = 0,
                'unexpected funder_award_id shape') AS shape_ok,
    assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok,
    assert_true(COUNT(*) = (SELECT COUNT(*) FROM openalex.awards.nci_australia_raw), 'rows lost in transform') AS complete_ok
FROM openalex.awards.nci_australia_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nci_merit_allocations' AND priority = 554;

-- Insert into openalex_awards_raw with priority.
-- Priority 554: direct-from-funder ingest of NCI's published NCMAS + ANUMAS allocation outcomes (in-kind).
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    554 as priority
FROM openalex.awards.nci_australia_awards;

## Verification Queries

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.nci_australia_awards
GROUP BY 1;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, start_year, end_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name,
       LEFT(description, 300) AS description_head
FROM openalex.awards.nci_australia_awards LIMIT 20;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 (NCMAS is annual, so a lead CI can hold one award per year).
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.nci_australia_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.nci_australia_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (in-kind: pct_amount 0 by design).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_affiliation
FROM openalex.awards.nci_australia_awards;

In [ ]:
%sql
SELECT start_year, funder_scheme, COUNT(*) AS n
FROM openalex.awards.nci_australia_awards
GROUP BY 1, 2 ORDER BY 1, 2;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nci_merit_allocations'
GROUP BY provenance, priority;